# **Privacy Layer Implementation**

### **Overview**
In this demonstration, we explore how to build a **Privacy Layer** for an AI agent — a fundamental step in ensuring *ethical, safe, and compliant* handling of user data within agentic systems.  
Before any user text is processed by a Large Language Model (LLM), the system applies an **input sanitization layer** that detects and masks personally identifiable information (PII) such as phone numbers, email addresses, or financial identifiers.  

This ensures that **no sensitive user information ever reaches the model context**, reducing risks of data leakage, misuse, or inadvertent memorization.

---

### **Objective**
The goal of this demo is to:
- **Detect and remove private entities** from user queries using regex-based pattern matching.  
- **Apply sanitization before inference**, ensuring models process only anonymized text.  
- **Illustrate ethical data handling practices** in line with Responsible AI and Data Privacy principles.

---

### **What the Demo Implements**

| Layer | Description | Implementation |
|--------|--------------|----------------|
| **Masking Function** | Uses pattern recognition to find and replace sensitive entities (phone, email, card, PAN, Aadhaar) with anonymized tokens. | Implemented as `mask_private_data()` using regular expressions. |
| **Pre-Processing Layer** | Acts as a protective middleware before the model call, sanitizing all incoming inputs. | Implemented as `sanitize_and_respond()` — it cleans the text before invoking the model. |

---

### **Why This Matters**
This privacy layer represents the **ethical foundation of AI governance** — embedding privacy-by-design directly into agent workflows.  
Without such filtering, LLMs could:
- Store or propagate private data across interactions.  
- Expose sensitive user details in future outputs.  
- Violate organizational data handling policies or legal compliance frameworks (GDPR, DPDP, HIPAA, etc.).

By integrating this masking layer, developers demonstrate **proactive adherence to Responsible AI principles** — ensuring that user trust and data safety are preserved at the system level.

---

### **Learning Outcome**
After completing this walkthrough, learners will be able to:
- Design privacy-preserving components in agent pipelines.  
- Understand how **ethical principles translate into technical safeguards**.  
- Extend this approach into larger **Responsible AI architectures** including safety filters, audit logs, and governance dashboards.


In [14]:
from dotenv import load_dotenv
load_dotenv()

True

In [15]:
# ==========================================================
# Import Libraries
# ==========================================================
import re
from langchain_openai import ChatOpenAI

# Initialize the model (using a lightweight version for demo)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [3]:
import re

pattern = r"^[A-Za-z]+(?:[ '-][A-Za-z]+)*$"

name = "Sahil Mattoo"

if re.match(pattern, name):
    print("Valid name")

Valid name


In [4]:
import re

def extract_names(sentence):
    # Regex for capitalized names (First Last format)
    pattern = r'\b[A-Z][a-z]+(?:\s[A-Z][a-z]+)*\b'
    
    names = re.findall(pattern, sentence)
    return names


# Example
sentence = "This is a test sentence, identify names like Rob, and Bob."

print(extract_names(sentence))

['This', 'Rob', 'Bob']


In [7]:
import re

def extract_phone_numbers(sentence):
    pattern = r'\b(?:\+?\d{1,3}[\s-]?)?[6-9]\d{9}\b'
    return re.findall(pattern, sentence)


text = "Call me at +91 9876543210 or 9123456789 or 1800-419-0025 tomorrow."
print(extract_phone_numbers(text))

['91 9876543210', '9123456789']


In [8]:
import re

def extract_phone_numbers(text):

    pattern = r'\b(?:\+?\d{1,3}[\s-]?)?(?:[6-9]\d{9}|1[0-9]{3}[-\s]?\d{3}[-\s]?\d{4})\b'

    return re.findall(pattern, text)


text = "Call 9876543210 or our toll free number 1800-419-0025."

print(extract_phone_numbers(text))

['9876543210', '1800-419-0025']


In [16]:
# ==========================================================
# Masking Function
# ==========================================================
def mask_private_data(text: str) -> str:
    """
    Detects and masks sensitive information like phone numbers,
    emails, credit card numbers, PAN, etc., before LLM processing.
    """
    # Phone numbers (India + international)
    text = re.sub(
        r'(?:(?:\+?91[\s\-]?)?(?:0)?)(?:[6-9]\d{9})\b|(?<!\d)\d{3}[\s\-]?\d{3}[\s\-]?\d{4}\b',
        "[PHONE]", text)
    
    # Email addresses
    text = re.sub(
        r'[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}',
        "[EMAIL]", text, flags=re.IGNORECASE)
    
    # Credit card numbers (13–19 digits)
    text = re.sub(
        r'\b(?:\d[ -]*?){13,19}\b',
        "[CARD]", text)
    
    # Indian PAN format
    text = re.sub(r'\b[A-Z]{5}\d{4}[A-Z]\b', "[PAN]", text)
    
    # Aadhaar-like 12-digit sequences
    text = re.sub(r'\b\d{12}\b', "[ID]", text)
    
    return text


# ==========================================================
# Pre-processing Layer
# ==========================================================
def sanitize_and_respond(user_input: str) -> str:
    """
    Pre-processing layer that sanitizes all incoming user text
    before passing to the model.
    """
    sanitized = mask_private_data(user_input)
    response = llm.invoke(sanitized)
    return f"Sanitized Input:\n{sanitized}\n\nModel Response:\n{response.content}"


# ==========================================================
# Demo Execution
# ==========================================================
if __name__ == "__main__":
    sample_inputs = [
        "Hi, my phone is 9876543210 and email is ankit@abc.com. What is Responsible AI?",
        "Send report for card 1234-5678-9876-5432 and PAN ABCDE1234F",
        "Aadhaar 999988887777 linked to my account; please explain privacy laws."
    ]
    
    for i, text in enumerate(sample_inputs, 1):
        print(f"\n--- Example {i} ---")
        print(sanitize_and_respond(text))


--- Example 1 ---
Sanitized Input:
Hi, my phone is [PHONE] and email is [EMAIL]. What is Responsible AI?

Model Response:
Responsible AI refers to the development and deployment of artificial intelligence systems in a manner that is ethical, transparent, and accountable. It encompasses a set of principles and practices aimed at ensuring that AI technologies are used in ways that are beneficial to society and do not cause harm. Key aspects of Responsible AI include:

1. **Fairness**: Ensuring that AI systems do not perpetuate or exacerbate biases and discrimination. This involves careful consideration of the data used to train AI models and the impact of their decisions on different groups.

2. **Transparency**: Making AI systems understandable and explainable to users and stakeholders. This includes providing insights into how decisions are made and the data that informs those decisions.

3. **Accountability**: Establishing clear lines of responsibility for the outcomes of AI systems.